# 02 · Amostra para rotulagem e pré-rótulo com FinBERT-PT-BR

Parte do `manchetes_filtradas.csv` gerado pelo notebook 01 e:

1. sorteia ~500 manchetes (até 25 por empresa);
2. classifica cada uma com o FinBERT-PT-BR (`lucas-leme/FinBERT-PT-BR`) como boa / ruim / neutra;
3. gera uma planilha única de rotulagem (`rotulagem.xlsx`).

**Rotulagem às cegas:** a planilha **não mostra** o pré-rótulo do FinBERT. Se a gente rotular vendo a resposta do modelo, tende a concordar com ele, e depois a avaliação do FinBERT contra os nossos rótulos fica inflada. O pré-rótulo fica guardado em `amostra_rotulagem.csv` e vira a linha de base do experimento.

**Como rodar no Colab:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*, depois *Executar tudo*.

In [ ]:
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

from pathlib import Path

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA = Path("/content/drive/MyDrive/portfolio-radar")
else:
    PASTA = Path("raw")

## Configuração

In [ ]:
POR_EMPRESA = 25          # 20 empresas x 25 = ~500 manchetes
INCLUIR_MULTIPLAS = False # manchetes que citam mais de uma empresa
SEMENTE = 42              # fixa o sorteio para dar sempre a mesma amostra

## Amostra

In [ ]:
import pandas as pd

filtradas = pd.read_csv(PASTA / "manchetes_filtradas.csv")
print(f"{len(filtradas)} manchetes filtradas")

if not INCLUIR_MULTIPLAS:
    filtradas = filtradas[~filtradas["empresas"].str.contains("|", regex=False)]
    print(f"{len(filtradas)} citam uma empresa só")

amostra = (
    filtradas.groupby("empresas", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), POR_EMPRESA), random_state=SEMENTE))
    .sample(frac=1, random_state=SEMENTE)  # embaralha
    .reset_index(drop=True)
)
amostra.insert(0, "id", [f"M{i:04d}" for i in range(1, len(amostra) + 1)])
print(f"Amostra: {len(amostra)} manchetes")
amostra["empresas"].value_counts().to_frame("manchetes").T

## Pré-rótulo com FinBERT-PT-BR

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("GPU" if device == 0 else "CPU (mais lento, mas funciona)")

clf = pipeline("text-classification", model="lucas-leme/FinBERT-PT-BR", device=device)
saidas = clf(amostra["manchete"].tolist(), batch_size=32, truncation=True)

ROTULO = {"POSITIVE": "boa", "NEGATIVE": "ruim", "NEUTRAL": "neutra"}
amostra["pre_rotulo"] = [ROTULO[s["label"]] for s in saidas]
amostra["confianca"] = [round(s["score"], 3) for s in saidas]
amostra["pre_rotulo"].value_counts().to_frame("manchetes")

## Planilha de rotulagem

In [ ]:
# Nome da empresa na planilha; sem o carteira.csv, fica só o ticker
arq_carteira = next((p for p in [Path("carteira.csv"), PASTA / "carteira.csv"] if p.exists()), None)
if arq_carteira:
    nomes = pd.read_csv(arq_carteira).set_index("ticker")["empresa"]
    amostra["empresa"] = amostra["empresas"].map(nomes)
else:
    amostra["empresa"] = amostra["empresas"]

amostra.to_csv(PASTA / "amostra_rotulagem.csv", index=False)

arq = PASTA / "rotulagem.xlsx"
amostra[["id", "empresa", "manchete", "link", "revisar"]].assign(rotulo="", observacao="").to_excel(arq, index=False)
print(f"{arq}: {len(amostra)} manchetes")

## Como rotular

Abra a `rotulagem.xlsx` (no Drive) com o Google Planilhas e preencha a coluna `rotulo` com:

| rótulo | quando usar |
|---|---|
| `boa` | impacto provável **positivo para o acionista** da empresa citada |
| `ruim` | impacto provável **negativo para o acionista** |
| `neutra` | sem impacto claro, ou só informativa |
| `descartar` | a manchete não é sobre a empresa (ex.: "Vale a pena…", Suzano a cidade) ou não está em português |

Linhas com `revisar = True` são as que mais provavelmente vão para `descartar`. Use `observacao` para dúvidas: a gente discute os casos difíceis antes de fechar o guia de rotulagem.